# 01. 市场状态分析

**目标**: 分析当前市场环境，判断市场状态
**数据来源**: JQData
**输出**: 市场状态报告、风险水平评估

In [4]:
import sys
sys.path.insert(0, '/home/taotao/dev/QuantTest/TRQuant')
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from notebooks.lib import get_jqdata_client, search_knowledge_base, save_research_conclusion
from notebooks.lib.viz_utils import plot_market_trend, plot_market_comparison, plot_returns_distribution

# 初始化JQData客户端
jq = get_jqdata_client()
print('✅ 环境加载完成')

ImportError: cannot import name 'plot_market_trend' from 'notebooks.lib.viz_utils' (/home/taotao/dev/QuantTest/TRQuant/notebooks/lib/viz_utils.py)

In [ ]:
# 获取主要指数数据（包含价格和成交量）
END_DATE = datetime.now().strftime('%Y-%m-%d')
START_DATE = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')
indices = {
    '000001.XSHG': '上证指数', 
    '399001.XSHE': '深证成指', 
    '399006.XSHE': '创业板指',
    '000300.XSHG': '沪深300'
}

index_data = {}  # 存储完整数据
index_close = {}  # 存储收盘价

for code, name in indices.items():
    try:
        df = jq.get_price(code, start_date=START_DATE, end_date=END_DATE, 
                         frequency='daily', fields=['open', 'high', 'low', 'close', 'volume'])
        if df is not None and not df.empty:
            index_data[name] = df
            index_close[name] = df['close']
            print(f'✅ {name} ({code}): {len(df)} 条记录')
    except Exception as e:
        print(f'❌ {name} ({code}) 获取失败: {e}')

index_df = pd.DataFrame(index_close)
print(f'\n成功获取 {len(index_df.columns)} 个指数的数据')
index_df.tail()

In [ ]:
# 市场统计计算
returns = index_df.pct_change().dropna()
stats = pd.DataFrame({
    '年化收益': returns.mean()*252,
    '年化波动': returns.std()*np.sqrt(252),
    '夏普比率': (returns.mean()*252)/(returns.std()*np.sqrt(252)),
})

print('=== 市场统计指标 ===')
stats.round(4)

## 可视化分析

### 1. 多指数对比图（归一化）

In [ ]:
# 绘制多指数对比图（归一化到起始值100）
fig = plot_market_comparison(index_close, normalize=True, title="主要指数对比（归一化）")
if fig:
    fig.show()
else:
    print("⚠️ 可视化工具不可用，请安装plotly或matplotlib")

### 2. 上证指数趋势分析（价格+均线+成交量）

In [ ]:
# 绘制上证指数趋势图
if '上证指数' in index_data:
    sh_data = index_data['上证指数']
    price_df = pd.DataFrame({'close': sh_data['close']})
    volume_series = sh_data['volume']
    
    fig = plot_market_trend(
        price_data=price_df,
        ma_periods=[5, 20, 60, 120],
        volume_data=volume_series,
        title="上证指数趋势分析"
    )
    if fig:
        fig.show()
else:
    print("⚠️ 上证指数数据不可用")

### 3. 收益率分布分析

In [ ]:
# 绘制收益率分布图（以沪深300为例）
if '沪深300' in index_close:
    hs300_returns = index_close['沪深300'].pct_change().dropna()
    fig = plot_returns_distribution(returns=hs300_returns, title="沪深300收益率分布")
    if fig:
        fig.show()
else:
    print("⚠️ 沪深300数据不可用")

### 4. 使用TrendAnalyzer进行深度分析

In [ ]:
# 使用项目中的TrendAnalyzer进行市场趋势分析
try:
    from core.trend_analyzer import TrendAnalyzer
    
    analyzer = TrendAnalyzer(jq_client=jq)
    result = analyzer.analyze_market(index_code="000001.XSHG")
    
    if result:
        print("=== 市场趋势分析结果 ===")
        print(f"市场阶段: {result.market_phase}")
        print(f"综合得分: {result.composite_score:.2f}")
        print(f"\n短期趋势: {result.short_term.score:.2f} ({result.short_term.direction.value})")
        print(f"中期趋势: {result.medium_term.score:.2f} ({result.medium_term.direction.value})")
        print(f"长期趋势: {result.long_term.score:.2f} ({result.long_term.direction.value})")
        
        # 获取仓位建议
        advice = analyzer.get_position_advice(result)
        print(f"\n=== 投资建议 ===")
        print(f"建议仓位: {advice['position']}")
        print(f"策略: {advice['strategy']}")
        print(f"推荐因子: {', '.join(advice['recommended_factors'])}")
    else:
        print("⚠️ 趋势分析返回空结果")
except Exception as e:
    print(f"⚠️ 趋势分析失败: {e}")

In [ ]:
# 保存研究结论
try:
    conclusion_data = {
        'date': END_DATE,
        'stats': stats.to_dict(),
        'indices_analyzed': list(index_df.columns),
    }
    
    # 如果进行了趋势分析，添加分析结果
    if 'result' in locals() and result:
        conclusion_data['trend_analysis'] = {
            'market_phase': result.market_phase,
            'composite_score': float(result.composite_score),
        }
    
    save_research_conclusion(
        module='market_analysis',
        findings=conclusion_data,
        recommendation=f"当前市场阶段: {result.market_phase if 'result' in locals() and result else '未分析'}",
        tags=['市场分析', '趋势分析']
    )
    print('✅ 研究结论已保存到知识库')
except Exception as e:
    print(f'⚠️ 保存结论失败: {e}')